Dummy Functions to Imitate the Functions in the Gescam Notebook

In [1]:
import numpy as np

def analyze_frame(file_path):
    """Dummy function to simulate frame analysis. Modeled after Frame 304 example."""
    individual_scores = [
        {"person_idx": i, "attention_score": score, "primary_object": obj, "head_bbox": [100+i*50, 100+i*50, 150+i*50, 150+i*50]}
        for i, (score, obj) in enumerate([
            (0.26, "Monitor/Screen"), (0.24, "Monitor/Screen"), (0.18, "Monitor/Screen"),
            (0.22, "Monitor/Screen"), (0.32, "Monitor/Screen"), (0.25, "Monitor/Screen"),
            (0.29, "Monitor/Screen"), (0.08, "Person/Teacher"), (0.26, "Monitor/Screen"),
            (0.04, "Person/Teacher"), (0.17, "Monitor/Screen"), (0.15, "Monitor/Screen"),
            (0.05, "Person/Teacher"), (0.0, "Person/Teacher")
        ])
    ]
    total_people = len(individual_scores)
    mean_attention = sum(score["attention_score"] for score in individual_scores) / total_people
    object_counts = {}
    for score in individual_scores:
        obj = score["primary_object"]
        object_counts[obj] = object_counts.get(obj, 0) + 1
    most_attended = max(object_counts.items(), key=lambda x: x[1])
    attention_distribution = {obj: count/total_people for obj, count in object_counts.items()}
    heatmap = np.random.uniform(0.0, 0.5, (10, 10)).tolist()
    return {
        "individual_scores": individual_scores,
        "frame_stats": {
            "total_people": total_people,
            "mean_attention": round(mean_attention, 2),
            "most_attended_object": [most_attended[0], most_attended[1]],
            "attention_distribution": {k: round(v, 3) for k, v in attention_distribution.items()}
        },
        "combined_heatmap": heatmap
    }

def compute_temporal_stats(frames_data):
    """Dummy function to simulate temporal analysis."""
    return {
        "frame_data": frames_data,
        "temporal_stats": {
            "attention_over_time": [frame["attention_data"]["frame_stats"]["mean_attention"] for frame in frames_data],
            "targets_over_time": [frame["attention_data"]["frame_stats"]["most_attended_object"] for frame in frames_data],
            "attention_stability": np.std([frame["attention_data"]["frame_stats"]["mean_attention"] for frame in frames_data]) if frames_data else 0.0,
            "attention_shifts": [
                {
                    "frame_id": frames_data[1]["frame_id"],
                    "from_object": frames_data[0]["attention_data"]["frame_stats"]["most_attended_object"],
                    "to_object": frames_data[1]["attention_data"]["frame_stats"]["most_attended_object"]
                }
            ] if len(frames_data) > 1 else []
        }
    }

def visualize_individual_attention(frame_img, attention_data, object_names=None, save_path=None):
    """Dummy function to simulate visualization."""
    if save_path is None:
        save_path = "/content/attention_test_results/dummy_attention.png"
    with open(save_path, 'w') as f:
        f.write("Dummy visualization for attention data")
    return save_path

Endpoints that utilize this dummy functions

All endpoints are POST functions


*   /upload_frame - is used to upload a frame for analysis
*   /analyze_frame - returns the analysis results the model generates
*   /analyze_temporal - returns analyzed results
*   /visualize_frame - returns the path to the visualization




In [4]:
# Kill any running ngrok process first
!pkill ngrok
!pip install flask pyngrok
from pyngrok import ngrok
!ngrok authtoken 207Efj2NVQAgP1LM7aqMDL7AKkW_6T4MTzNgKcpZiZQEcBzAn
from flask import Flask, request, jsonify
import os
from werkzeug.utils import secure_filename
import numpy as np
from threading import Thread
import uuid

app = Flask(__name__)

# Configuration
UPLOAD_FOLDER = '/content/uploads'
RESULT_FOLDER = '/content/attention_test_results'
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg'}
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['RESULT_FOLDER'] = RESULT_FOLDER

# Ensure directories exist
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(RESULT_FOLDER, exist_ok=True)

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

@app.route('/upload_frame', methods=['POST'])
def upload_frame():
    """Endpoint to upload a frame image."""
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400
    file = request.files['file']
    if file.filename == '':
        return jsonify({"error": "No selected file"}), 400
    if file and allowed_file(file.filename):
        filename = secure_filename(file.filename)
        unique_filename = f"{uuid.uuid4()}_{filename}"
        file_path = os.path.join(app.config['UPLOAD_FOLDER'], unique_filename)
        file.save(file_path)
        return jsonify({"message": "File uploaded successfully", "file_path": file_path}), 200
    return jsonify({"error": "Invalid file type"}), 400

@app.route('/analyze_frame', methods=['POST'])
def analyze_frame_endpoint():
    """Endpoint to analyze a single frame using dummy analyze_frame."""
    data = request.get_json()
    if not data or 'file_path' not in data:
        return jsonify({"error": "file_path is required"}), 400
    file_path = data['file_path']
    if not os.path.exists(file_path):
        return jsonify({"error": "File does not exist"}), 404
    try:
        result = analyze_frame(file_path)
        return jsonify(result), 200
    except Exception as e:
        return jsonify({"error": f"Analysis failed: {str(e)}"}), 500

@app.route('/analyze_temporal', methods=['POST'])
def analyze_temporal_endpoint():
    """Endpoint for temporal analysis using dummy compute_temporal_stats."""
    data = request.get_json()
    if not data or 'frames_data' not in data:
        return jsonify({"error": "frames_data is required"}), 400
    try:
        result = compute_temporal_stats(data['frames_data'])
        return jsonify(result), 200
    except Exception as e:
        return jsonify({"error": f"Temporal analysis failed: {str(e)}"}), 500

@app.route('/visualize_frame', methods=['POST'])
def visualize_frame_endpoint():
    """Endpoint to visualize frame attention using dummy visualize_individual_attention."""
    data = request.get_json()
    if not data or 'file_path' not in data:
        return jsonify({"error": "file_path is required"}), 400
    file_path = data['file_path']
    if not os.path.exists(file_path):
        return jsonify({"error": "File does not exist"}), 404
    try:
        attention_data = analyze_frame(file_path)
        frame_img = np.zeros((1080, 1920, 3))
        vis_path = visualize_individual_attention(
            frame_img=frame_img,
            attention_data=attention_data,
            save_path=os.path.join(app.config['RESULT_FOLDER'], f"attention_{uuid.uuid4()}.png")
        )
        return jsonify({"message": "Visualization created", "vis_path": vis_path}), 200
    except Exception as e:
        return jsonify({"error": f"Visualization failed: {str(e)}"}), 500

# Run Flask app in a background thread
def run_app():
    app.run(port=5000)

# Start ngrok tunnel
public_url = ngrok.connect(5000).public_url
print(f" * ngrok tunnel available at: {public_url}")

# Start Flask app
thread = Thread(target=run_app)
thread.start()

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
 * ngrok tunnel available at: https://7841-34-83-220-117.ngrok-free.app
 * Serving Flask app '__main__'
 * Debug mode: off


Endpoints that interact with the main notebook (Same format as the endpoints above)

In [ ]:
# Install import_ipynb to import the main notebook
!pip install import_ipynb
import import_ipynb
import os

# Run the main notebook to load its definitions
# Ensure MS GESCAM.ipynb is uploaded to Colab
if os.path.exists('/content/MS GESCAM.ipynb'):
    %run '/content/MS GESCAM.ipynb'
else:
    print("Error: MS GESCAM.ipynb not found. Please upload it to /content/.")

# Kill any running ngrok process first
!pkill ngrok
!pip install flask pyngrok
from pyngrok import ngrok
!ngrok authtoken 207Efj2NVQAgP1LM7aqMDL7AKkW_6T4MTzNgKcpZiZQEcBzAn
from flask import Flask, request, jsonify
import os
from werkzeug.utils import secure_filename
import numpy as np
from threading import Thread
import uuid
import torch
from PIL import Image
import torchvision.transforms as transforms

app = Flask(__name__)

# Configuration
UPLOAD_FOLDER = '/content/uploads'
RESULT_FOLDER = '/content/attention_test_results'
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg'}
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['RESULT_FOLDER'] = RESULT_FOLDER

# Ensure directories exist
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(RESULT_FOLDER, exist_ok=True)

# Initialize model and device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model
model_path = "/content/best_model.pt"
print("Loading model...")
try:
    model = MSGESCAMModel(pretrained=False, output_size=64)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("Loaded model checkpoint successfully")
except Exception as e:
    print(f"Error loading model: {e}")
    raise Exception("Model loading failed. Ensure best_model.pt is uploaded.")
model = model.to(device)
model.eval()

# Image transform
try:
    transform = get_transforms(augment=False)
except NameError:
    raise Exception("get_transforms not defined. Ensure MS GESCAM.ipynb is loaded.")

# Object class names
object_names = {
    0: "Person/Teacher",
    1: "Board",
    2: "Book/Notebook",
    3: "Monitor/Screen",
    4: "Phone",
    5: "Desk/Table",
    6: "Water Dispenser",
    7: "Mug",
    8: "Lamp",
    9: "Other1",
    10: "Other2"
}

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

def analyze_frame(file_path):
    """Analyze a frame using analyze_frame_attention from MS GESCAM.ipynb."""
    try:
        img = Image.open(file_path).convert('RGB')
        img_tensor = transform(img).unsqueeze(0).to(device)
        frame_data = [
            (
                img_tensor[0],
                torch.zeros(1, 4).to(device),
                torch.zeros(1, 2).to(device),
                torch.zeros(1).to(device),
                torch.zeros(1).to(device),
                torch.zeros(1, 4).to(device),
                torch.zeros(1).to(device),
                torch.ones(1).to(device),
                {'frame_id': 0, 'person_id': 0}
            )
        ]
        try:
            attention_data = analyze_frame_attention(
                model=model,
                frame_data=frame_data,
                device=device,
                context="lecture"
            )
        except NameError:
            raise Exception("analyze_frame_attention not defined. Ensure MS GESCAM.ipynb is loaded.")
        if 'combined_heatmap' in attention_data:
            attention_data['combined_heatmap'] = np.array(attention_data['combined_heatmap']).tolist()
        return attention_data
    except Exception as e:
        raise Exception(f"Frame analysis failed: {str(e)}")

def compute_temporal_stats(frames_data):
    """Compute temporal stats, adapted from test_attention_scoring."""
    try:
        temporal_stats = {
            'attention_over_time': [],
            'targets_over_time': [],
            'attention_shifts': []
        }
        for frame in frames_data:
            frame_stats = frame['attention_data']['frame_stats']
            temporal_stats['attention_over_time'].append(frame_stats['mean_attention'])
            temporal_stats['targets_over_time'].append(frame_stats['most_attended_object'])
        temporal_stats['attention_stability'] = np.std(temporal_stats['attention_over_time']) if temporal_stats['attention_over_time'] else 0.0
        for i in range(1, len(frames_data)):
            prev = frames_data[i-1]['attention_data']['frame_stats']['most_attended_object']
            curr = frames_data[i]['attention_data']['frame_stats']['most_attended_object']
            if prev and curr and prev[0] != curr[0]:
                temporal_stats['attention_shifts'].append({
                    'frame_id': frames_data[i]['frame_id'],
                    'from_object': prev,
                    'to_object': curr
                })
        return {
            'frame_data': frames_data,
            'temporal_stats': temporal_stats
        }
    except Exception as e:
        raise Exception(f"Temporal stats computation failed: {str(e)}")

def visualize_individual_attention(frame_img, attention_data, object_names=object_names, save_path=None):
    """Visualize attention using visualize_individual_attention from MS GESCAM.ipynb."""
    try:
        if save_path is None:
            save_path = os.path.join(app.config['RESULT_FOLDER'], f"attention_{uuid.uuid4()}.png")
        if isinstance(frame_img, torch.Tensor):
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            img_vis = frame_img.clone()
            img_vis = img_vis * std + mean
            img_vis = img_vis.permute(1, 2, 0).numpy()
            img_vis = np.clip(img_vis, 0, 1)
        else:
            img_vis = frame_img
        try:
            visualize_individual_attention(
                frame_img=img_vis,
                attention_data=attention_data,
                object_names=object_names,
                save_path=save_path
            )
        except NameError:
            raise Exception("visualize_individual_attention not defined. Ensure MS GESCAM.ipynb is loaded.")
        return save_path
    except Exception as e:
        raise Exception(f"Visualization failed: {str(e)}")

@app.route('/upload_frame', methods=['POST'])
def upload_frame():
    """Endpoint to upload a frame image."""
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400
    file = request.files['file']
    if file.filename == '':
        return jsonify({"error": "No selected file"}), 400
    if file and allowed_file(file.filename):
        filename = secure_filename(file.filename)
        unique_filename = f"{uuid.uuid4()}_{filename}"
        file_path = os.path.join(app.config['UPLOAD_FOLDER'], unique_filename)
        file.save(file_path)
        return jsonify({"message": "File uploaded successfully", "file_path": file_path}), 200
    return jsonify({"error": "Invalid file type"}), 400

@app.route('/analyze_frame', methods=['POST'])
def analyze_frame_endpoint():
    """Endpoint to analyze a frame using MS GESCAM.ipynb."""
    data = request.get_json()
    if not data or 'file_path' not in data:
        return jsonify({"error": "file_path is required"}), 400
    file_path = data['file_path']
    if not os.path.exists(file_path):
        return jsonify({"error": "File does not exist"}), 404
    try:
        result = analyze_frame(file_path)
        return jsonify(result), 200
    except Exception as e:
        return jsonify({"error": f"Analysis failed: {str(e)}"}), 500

@app.route('/analyze_temporal', methods=['POST'])
def analyze_temporal_endpoint():
    """Endpoint for temporal analysis."""
    data = request.get_json()
    if not data or 'frames_data' not in data:
        return jsonify({"error": "frames_data is required"}), 400
    try:
        result = compute_temporal_stats(data['frames_data'])
        return jsonify(result), 200
    except Exception as e:
        return jsonify({"error": f"Temporal analysis failed: {str(e)}"}), 500

@app.route('/visualize_frame', methods=['POST'])
def visualize_frame_endpoint():
    """Endpoint to visualize frame attention using MS GESCAM.ipynb."""
    data = request.get_json()
    if not data or 'file_path' not in data:
        return jsonify({"error": "file_path is required"}), 400
    file_path = data['file_path']
    if not os.path.exists(file_path):
        return jsonify({"error": "File does not exist"}), 404
    try:
        attention_data = analyze_frame(file_path)
        img = Image.open(file_path).convert('RGB')
        img_tensor = transform(img).to(device)
        vis_path = visualize_individual_attention(
            frame_img=img_tensor,
            attention_data=attention_data,
            save_path=os.path.join(app.config['RESULT_FOLDER'], f"attention_{uuid.uuid4()}.png")
        )
        return jsonify({"message": "Visualization created", "vis_path": vis_path}), 200
    except Exception as e:
        return jsonify({"error": f"Visualization failed: {str(e)}"}), 500

# Run Flask app in a background thread
def run_app():
    app.run(port=5000)

# Start ngrok tunnel
public_url = ngrok.connect(5000).public_url
print(f" * ngrok tunnel available at: {public_url}")

# Start Flask app
thread = Thread(target=run_app)
thread.start()